# Notebook 02 — Feature Engineering

**Goal:** Transform the raw dataset into a rich feature matrix for modelling.

**Inputs:** `data/raw/train.csv`, `data/raw/test.csv`  
**Outputs:** `data/processed/train_features.parquet`, `data/processed/test_features.parquet`

**No-leakage rule:** For lap L, only information available at the *start* of lap L may be used. Lag/rolling features are grouped within `(Race, Year, Driver, Stint)` so they never cross stint or driver boundaries. (`Stint` resets to 1 for every driver after each pit stop — grouping without `Driver` would mix all drivers' laps together within the same stint number.)

**Target encoding note:** `Race_target_encoded`, `Driver_target_encoded`, and `Driver_avg_stint_length` are **not** saved to parquet — they must be computed inside each CV fold (Notebook 05) using only that fold's training rows. The helper `apply_target_encodings()` is defined here and redefined inline in every downstream notebook.

## Feature Map
| # | Category | Features |
|---|----------|----------|
| 1 | Tyre age | `TyreLife_normalized_by_compound`, `TyreLife_sq` |
| 2 | Compound | `is_wet_tyre`, `compound_ordinal` |
| 3 | Race context | `laps_remaining`, `is_testing_session` |
| 4 | Raw context | `Stint`, `Position` |
| 5 | Degradation | `Cumulative_Degradation_winsorized`, `Degradation_rate`, `Degradation_acceleration` |
| 6 | Lag | `LapTime_lag{1,2,3}`, `LapTime_Delta_lag{1,2,3}` |
| 7 | Rolling | `LapTime_rolling_mean_{3,5}`, `LapTime_rolling_std_{3,5}`, `Degradation_rolling_slope_{3,5}` |
| 8 | Interaction | `TyreLife_x_laps_remaining`, `Degradation_x_RaceProgress`, `Position_x_RaceProgress` |
| 9 | Target encoding | `Race_target_encoded`, `Driver_target_encoded`, `Driver_avg_stint_length` (fold-aware only) |

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

# --- Robust project root detection (works from workspace root or notebooks/) ---
cwd = Path.cwd()
if cwd.name == 'notebooks' or 'notebooks' in str(cwd):
    while cwd.name != 'predict-f1-pit-stops' and cwd.parent != cwd:
        cwd = cwd.parent
project_root  = cwd
data_dir      = project_root / 'data' / 'raw'
processed_dir = project_root / 'data' / 'processed'
processed_dir.mkdir(parents=True, exist_ok=True)

assert data_dir.exists(), f'Data directory not found: {data_dir}'
print(f'Project root : {project_root}')
print(f'Processed dir: {processed_dir}')

Project root : c:\Repos\predict-f1-pit-stops
Processed dir: c:\Repos\predict-f1-pit-stops\data\processed


In [2]:
train_raw = pd.read_csv(data_dir / 'train.csv')
test_raw  = pd.read_csv(data_dir / 'test.csv')

print(f'Train: {train_raw.shape}  |  Test: {test_raw.shape}')
print(f'Columns: {list(train_raw.columns)}')

Train: (439140, 16)  |  Test: (188165, 15)
Columns: ['id', 'Driver', 'Compound', 'Race', 'Year', 'PitStop', 'LapNumber', 'Stint', 'TyreLife', 'Position', 'LapTime (s)', 'LapTime_Delta', 'Cumulative_Degradation', 'RaceProgress', 'Position_Change', 'PitNextLap']


## 1. Tyre Age Features

**`TyreLife_normalized_by_compound`** — the single most important feature motivated by EDA.

Raw `TyreLife` is not comparable across compounds: a SOFT tyre at lap 10 is past its cliff (cliff ≈ 13), while a HARD tyre at lap 10 is only 16% of the way to its cliff (cliff ≈ 61). Dividing by the compound-specific cliff threshold creates a normalized age on a common scale where 1.0 = tyre has reached peak pit probability.

Cliff thresholds from EDA S-curves (TyreLife where P(PitNextLap) peaks):
- SOFT: 13 laps, MEDIUM: 49 laps, HARD: 61 laps
- INTERMEDIATE/WET: set to 1 (wet tyres are replaced for non-durability reasons; normalization is meaningless)

**`TyreLife_sq`** — captures the nonlinear tyre cliff (S-curve) that linear models miss. As TyreLife grows, P(pit) initially rises slowly then accelerates. The squared term lets the model represent this curvature.

In [3]:
# Compound-specific cliff thresholds from EDA S-curves
CLIFF_THRESHOLDS = {'SOFT': 13, 'MEDIUM': 49, 'HARD': 61, 'INTERMEDIATE': 1, 'WET': 1}

# Dry-compound durability ordinal (wet compounds get 0 — separate is_wet_tyre flag)
COMPOUND_ORDINAL  = {'SOFT': 1, 'MEDIUM': 2, 'HARD': 3, 'INTERMEDIATE': 0, 'WET': 0}

# Winsorization bounds for Cumulative_Degradation (1st/99th pct from EDA)
WINSOR_LOW, WINSOR_HIGH = -205, 122

# Demonstrate tyre age normalization on a small slice
demo = train_raw[['Compound', 'TyreLife']].head(20).copy()
demo['cliff'] = demo['Compound'].map(CLIFF_THRESHOLDS)
demo['TyreLife_normalized_by_compound'] = demo['TyreLife'] / demo['cliff']
print('Tyre age normalization demo:')
print(demo.groupby('Compound')[['TyreLife', 'TyreLife_normalized_by_compound']].describe().round(3))

# This is intentional but worth being aware of:
# INTERMEDIATE/WET normalized values will be very large outliers (up to 71.0 per the stats cell). GBMs handle this fine since is_wet_tyre=1 will condition the splits, but it's a quirk of the encoding choice.

Tyre age normalization demo:
             TyreLife                                                \
                count    mean     std   min   25%   50%   75%   max   
Compound                                                              
HARD             11.0  14.455  10.885   2.0   7.5  10.0  20.0  39.0   
INTERMEDIATE      1.0  12.000     NaN  12.0  12.0  12.0  12.0  12.0   
MEDIUM            8.0   7.375   6.589   1.0   2.0   4.5  13.0  17.0   

             TyreLife_normalized_by_compound                                 \
                                       count    mean    std     min     25%   
Compound                                                                      
HARD                                    11.0   0.237  0.178   0.033   0.123   
INTERMEDIATE                             1.0  12.000    NaN  12.000  12.000   
MEDIUM                                   8.0   0.151  0.134   0.020   0.041   

                                      
                 50%     75%  

## 2. Compound Encoding

**`is_wet_tyre`** — binary flag for INTERMEDIATE and WET compounds. These are used in wet conditions and replaced when track conditions change, not when tyres degrade. They follow a completely different decision process from dry tyres — the durability ordinal is meaningless for them.

**`compound_ordinal`** — durability ordering for dry compounds only: SOFT=1 (least durable), MEDIUM=2, HARD=3 (most durable). Wet compounds get 0. This encodes the physical durability axis without assuming wet compounds fit on the same scale.

Why not one-hot encode? With 5 compounds and ~420K rows, the ordinal encoding is sufficient for tree-based models and preserves the durability ordering signal. One-hot would add 4 columns with the same information.

In [4]:
# Compound distribution check
print('Compound value counts in train:')
cmp_stats = (train_raw.groupby('Compound')
             .agg(rows=('PitNextLap', 'count'), pit_rate=('PitNextLap', 'mean'))
             .assign(ordinal=lambda d: d.index.map(COMPOUND_ORDINAL),
                     is_wet=lambda d: d.index.isin({'INTERMEDIATE', 'WET'}).astype(int),
                     cliff=lambda d: d.index.map(CLIFF_THRESHOLDS))
             .sort_values('ordinal'))
print(cmp_stats.round(3))

Compound value counts in train:
                rows  pit_rate  ordinal  is_wet  cliff
Compound                                              
INTERMEDIATE   17382     0.152        0       1      1
WET             1355     0.025        0       1      1
SOFT           38744     0.193        1       0     13
MEDIUM        211141     0.101        2       0     49
HARD          170518     0.328        3       0     61


## 3. Race Context Features

**`laps_remaining`** = `1 - RaceProgress` — EDA showed a 1.8× drop in pit probability once RaceProgress exceeds 0.80 (20.5% → 11.2%). Expressing this as laps *remaining* rather than laps *completed* gives the model a more direct signal: as remaining time shrinks, pitting becomes strategically irrational.

**`is_testing_session`** — Pre-Season Testing rows (22,492 total, ~5% of train) have a 14.7% pit rate vs 20.2% for actual races. These are practice sessions with very different tyre management patterns. The flag lets the model learn distinct behaviour for testing without mixing it with race dynamics. An ablation in Notebook 05 will test whether excluding them improves CV AUC.

In [5]:
print('RaceProgress distribution (late-race pit suppression):')
bins = [0, 0.5, 0.7, 0.8, 0.9, 1.0]
labels = ['0-50%', '50-70%', '70-80%', '80-90%', '90-100%']
bucket = pd.cut(train_raw['RaceProgress'], bins=bins, labels=labels)
print(train_raw.groupby(bucket, observed=True)['PitNextLap'].agg(['mean', 'count']).round(3))

print('\nPre-Season Testing rows:')
testing_mask = train_raw['Race'] == 'Pre-Season Testing'
print(f'  Rows: {testing_mask.sum():,}  ({testing_mask.mean()*100:.1f}% of train)')
print(f'  Pit rate: {train_raw.loc[testing_mask, "PitNextLap"].mean():.3f} (vs {train_raw.loc[~testing_mask, "PitNextLap"].mean():.3f} for races)')

# Findings:
# The 50–70% band is the main F1 pit window — first stops typically open here.
# The relationship is an inverted-U, not monotonically declining. The laps_remaining feature still captures the late-race suppression, but the tree will need to discover the mid-race peak through splits on RaceProgress directly.

RaceProgress distribution (late-race pit suppression):
               mean   count
RaceProgress               
0-50%         0.165  326823
50-70%        0.387   63180
70-80%        0.287   19420
80-90%        0.149   18430
90-100%       0.051   11287

Pre-Season Testing rows:
  Rows: 22,492  (5.1% of train)
  Pit rate: 0.147 (vs 0.202 for races)


## 4. Degradation Features

**`Cumulative_Degradation_winsorized`** — `Cumulative_Degradation` has extreme outliers (min=−274, max=+2,412 from EDA). Winsorizing at the 1st/99th percentile (≈−205 / +122) prevents these from dominating linear models and reduces noise in interaction features.

**`Degradation_rate`** = winsorized degradation / TyreLife — normalizes total degradation by stint age. Two stints with the same cumulative degradation but different lengths have very different degradation trajectories; the rate captures this.

**`Degradation_acceleration`** = lap-over-lap delta of `Cumulative_Degradation`, grouped within `(Race, Year, Stint)`. This is effectively `LapTime_Delta` itself (since `Cumulative_Degradation` is its cumulative sum), but expressing it as acceleration makes the *rate of change of degradation* explicit. The first lap of each stint gets 0 (no prior lap in the same stint).

In [6]:
print(f'Cumulative_Degradation percentiles:')
print(train_raw['Cumulative_Degradation'].describe(percentiles=[.01, .05, .25, .5, .75, .95, .99]).round(2))

# Verify winsorization bounds
p01 = train_raw['Cumulative_Degradation'].quantile(0.01)
p99 = train_raw['Cumulative_Degradation'].quantile(0.99)
print(f'\n1st pct: {p01:.1f}  (WINSOR_LOW={WINSOR_LOW})')
print(f'99th pct: {p99:.1f}  (WINSOR_HIGH={WINSOR_HIGH})')

Cumulative_Degradation percentiles:
count    439140.00
mean        -25.72
std          54.77
min        -274.56
1%         -205.03
5%         -104.79
25%         -46.57
50%         -20.99
75%          -6.20
95%          84.40
99%         122.15
max        2412.03
Name: Cumulative_Degradation, dtype: float64

1st pct: -205.0  (WINSOR_LOW=-205)
99th pct: 122.2  (WINSOR_HIGH=122)


## 5. Lag Features

ACF analysis in Notebook 01 showed significant positive autocorrelation at lag-1 in 69.8% of stints (median ACF=0.035). This justifies including the previous 1–3 laps' values as features — the model can see whether lap times are deteriorating, stable, or recovering.

**Grouping within `(Race, Year, Driver, Stint)`** is essential for two reasons:
1. **Stint boundary**: lags must not cross into the previous stint (different tyres, fresh degradation counter).
2. **Driver boundary**: `Stint` is a per-driver counter — at the start of the race, every driver is on Stint=1. Grouping without `Driver` would mix all 20 drivers' first-stint laps together and assign one driver's lap time as another driver's lag. Adding `Driver` ensures each driver's laps form an isolated time series.

The `.shift()` on a grouped series produces NaN at the boundary (first lap of each group), which we fill with 0 — a neutral "no prior information" value.

**Why 3 lags?** ACF typically decays to near-zero by lag 3–5 for lap time data. Beyond lag 3, the added signal is minimal and the features become sparser (more NaN-fills at stint start). We cap at 3 to balance coverage and parsimony.

In [7]:
# Illustrate the Driver groupby requirement
# Without Driver: (Race, Year, Stint=1) mixes all 20 drivers' first-stint laps
# With Driver:    each driver's first stint is an isolated group → lag=NaN at stint start

tmp = train_raw.sort_values(['Race', 'Year', 'LapNumber']).head(200)[
    ['Race', 'Year', 'Driver', 'Stint', 'TyreLife', 'LapTime (s)']
].copy()

# WRONG groupby — missing Driver
tmp['lag1_wrong'] = tmp.groupby(['Race', 'Year', 'Stint'])['LapTime (s)'].shift(1)
# CORRECT groupby — includes Driver
tmp['lag1_correct'] = tmp.groupby(['Race', 'Year', 'Driver', 'Stint'])['LapTime (s)'].shift(1)

# Show stint-start rows (TyreLife=1) where the two disagree
stint_starts = tmp[tmp['TyreLife'] == 1.0].head(8)
print('Stint starts (TyreLife=1) — correct lag should be NaN, wrong lag borrows from another driver:')
print(stint_starts[['Driver', 'Stint', 'TyreLife', 'LapTime (s)', 'lag1_wrong', 'lag1_correct']].to_string())

Stint starts (TyreLife=1) — correct lag should be NaN, wrong lag borrows from another driver:
       Driver  Stint  TyreLife  LapTime (s)  lag1_wrong  lag1_correct
7544      ALO      1       1.0       89.929         NaN           NaN
31758     LAT      1       1.0       98.329      89.929           NaN
73559     WEB      1       1.0       99.918      98.329           NaN
309725    ALB      1       1.0      116.433      98.913           NaN
320314    PER      3       1.0      110.580      88.369           NaN
3216     D113      1       1.0       94.113         NaN           NaN
4155     D222      1       1.0      100.919      94.113           NaN
5283      CAR      1       1.0       90.926     100.919           NaN


## 6. Rolling Statistics

Rolling windows smooth out single-lap noise and capture local trends that lags alone can't express.

**`LapTime_rolling_mean_{3,5}`** — local pace baseline. A rising mean (relative to the driver's overall reference pace) signals tyre degradation. `min_periods=1` means the first lap in a stint gets the window filled with whatever observations are available — no NaNs at stint start.

**`LapTime_rolling_std_{3,5}`** — variance of recent lap times. High variance can indicate inconsistent conditions (traffic, track evolution) or an erratic degradation profile. `min_periods=2` to avoid std of a single point; first lap gets 0.

**`Degradation_rolling_slope_{3,5}`** — slope of the degradation curve over the last w laps, estimated as `(current − value w−1 laps ago) / (w−1)`. This is a finite-difference linear slope. A steep positive slope (rapidly worsening degradation) is a strong pit signal.

All rolling operations use the same `(Race, Year, Driver, Stint)` groupby as the lag features — same two reasons apply.

In [8]:
# Illustrate rolling slope calculation on a sample stint
sample_stint = (train_raw
                .sort_values(['Race', 'Year', 'LapNumber'])
                .groupby(['Race', 'Year', 'Driver', 'Stint'])
                .filter(lambda g: len(g) >= 10)
                .groupby(['Race', 'Year', 'Driver', 'Stint'])
                .first()
                .reset_index()
                .iloc[0])

mask = ((train_raw['Race'] == sample_stint['Race']) &
        (train_raw['Year'] == sample_stint['Year']) &
        (train_raw['Driver'] == sample_stint['Driver']) &
        (train_raw['Stint'] == sample_stint['Stint']))

stint_df = train_raw[mask].sort_values('LapNumber')[['LapNumber', 'TyreLife', 'Cumulative_Degradation']].copy()
stint_df['Deg_slope_3'] = (stint_df['Cumulative_Degradation'] - stint_df['Cumulative_Degradation'].shift(2)) / 2
print(f'Sample stint: Race={sample_stint["Race"]}, Driver={sample_stint["Driver"]}')
print(stint_df.head(10).to_string(index=False))

Sample stint: Race=Abu Dhabi Grand Prix, Driver=ALE
 LapNumber  TyreLife  Cumulative_Degradation  Deg_slope_3
        12       1.0                   0.000          NaN
        16       3.0                 -19.634          NaN
        18       4.0                 -21.280     -10.6400
        19       6.0                 -20.897      -0.6315
        20       7.0                 -19.639       0.8205
        21       8.0                 -20.971      -0.0370
        24      12.0                 -19.800      -0.0805
        28      16.0                 -19.736       0.6175
        32      17.0                 -20.268      -0.2340
        51      28.0                 -21.380      -0.8220


## 7. Interaction Features

Tree models discover interactions automatically, but including explicit interaction features reduces the depth needed to capture them, which regularises the model and speeds training.

**`TyreLife_x_laps_remaining`** — the joint effect of tyre age and race time remaining. High TyreLife *and* lots of laps remaining → tyre must be pitted soon. High TyreLife *but* few laps remaining → can nurse the tyres to the end. Neither variable alone captures this strategic trade-off.

**`Degradation_x_RaceProgress`** — degradation matters more as the race progresses. Early in a race, drivers tolerate higher degradation to maintain track position; late in a race, any degradation spike triggers an immediate pit decision. This product encodes that context-dependence.

**`Position_x_RaceProgress`** — a driver in P1 late in the race will avoid pitting to protect their lead (negative signal); a driver in P15 late in the race has little to lose from pitting (neutral/positive signal). The interaction captures this asymmetry.

In [9]:
# Correlation of each interaction with target
tmp = train_raw.copy()
tmp['TyreLife_x_laps_remaining']     = tmp['TyreLife'] * (1 - tmp['RaceProgress'])
tmp['Degradation_x_RaceProgress']    = tmp['Cumulative_Degradation'].clip(WINSOR_LOW, WINSOR_HIGH) * tmp['RaceProgress']
tmp['Position_x_RaceProgress']       = tmp['Position'] * tmp['RaceProgress']

interaction_cols = ['TyreLife_x_laps_remaining', 'Degradation_x_RaceProgress', 'Position_x_RaceProgress']
print('Interaction feature correlations with PitNextLap:')
print(tmp[interaction_cols + ['PitNextLap']].corr()['PitNextLap'].drop('PitNextLap').round(4))

Interaction feature correlations with PitNextLap:
TyreLife_x_laps_remaining     0.1983
Degradation_x_RaceProgress   -0.2463
Position_x_RaceProgress       0.1442
Name: PitNextLap, dtype: float64


## 8. `build_base_features()` — Consolidated Function

All feature engineering above is assembled into a single, self-contained function. This function is **redefined inline in every downstream notebook** (05, 10) so those notebooks can run on Kaggle without importing any local modules.

The function:
1. Accepts a raw DataFrame (train or test)
2. Sorts by `(Race, Year, LapNumber)` to guarantee temporal order
3. Builds all 26 base features in order
4. Returns the enriched DataFrame (original columns preserved)

Target encodings are **excluded** — see `apply_target_encodings()` below.

In [10]:
def build_base_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Build all non-leaky base features from a raw train/test DataFrame.
    Sorts by (Race, Year, LapNumber) internally.
    Target encodings are NOT included — use apply_target_encodings() inside CV folds.
    """
    CLIFF   = {'SOFT': 13, 'MEDIUM': 49, 'HARD': 61, 'INTERMEDIATE': 1, 'WET': 1}
    ORDINAL = {'SOFT':  1, 'MEDIUM':  2, 'HARD':  3, 'INTERMEDIATE': 0, 'WET': 0}
    W_LO, W_HI = -205, 122
    STINT_KEYS = ['Race', 'Year', 'Driver', 'Stint']

    df = df.copy().sort_values(['Race', 'Year', 'LapNumber']).reset_index(drop=True)

    # ── 1. Tyre age ───────────────────────────────────────────────────────────
    df['TyreLife_normalized_by_compound'] = df['TyreLife'] / df['Compound'].map(CLIFF)
    df['TyreLife_sq']                     = df['TyreLife'] ** 2

    # ── 2. Compound encoding ──────────────────────────────────────────────────
    df['is_wet_tyre']      = df['Compound'].isin({'INTERMEDIATE', 'WET'}).astype(np.int8)
    df['compound_ordinal'] = df['Compound'].map(ORDINAL).astype(np.int8)

    # ── 3. Race context ───────────────────────────────────────────────────────
    df['laps_remaining']     = 1.0 - df['RaceProgress']
    df['is_testing_session'] = (df['Race'] == 'Pre-Season Testing').astype(np.int8)

    # ── 4. Degradation ────────────────────────────────────────────────────────
    df['Cumulative_Degradation_winsorized'] = df['Cumulative_Degradation'].clip(W_LO, W_HI)
    # Clip TyreLife at 1 to avoid division by zero on the very first lap
    df['Degradation_rate']        = df['Cumulative_Degradation_winsorized'] / df['TyreLife'].clip(lower=1)
    # groupby.diff() is C-level — fast even with many groups
    df['Degradation_acceleration'] = (
        df.groupby(STINT_KEYS)['Cumulative_Degradation']
          .diff(1)
          .fillna(0.0)
    )

    # ── 5. Lag features (within Driver's stint — NaN at boundary → fill 0) ───
    # Stint resets to 1 for each driver after every pit stop. Grouping without
    # Driver merges all drivers' Stint=1 laps, causing cross-driver contamination.
    # groupby.shift() is C-level — fast.
    for raw_col, feat_base in [
        ('LapTime (s)',   'LapTime'),
        ('LapTime_Delta', 'LapTime_Delta'),
    ]:
        grp = df.groupby(STINT_KEYS)[raw_col]
        for lag in [1, 2, 3]:
            df[f'{feat_base}_lag{lag}'] = grp.shift(lag).fillna(0.0)

    # ── 6. Rolling stats (within Driver's stint) ─────────────────────────────
    # Use groupby().rolling() (C-level) rather than transform(lambda x: x.rolling(...))
    # (Python-level). With ~2500 groups the lambda approach is ~25× slower.
    # groupby().rolling() returns a MultiIndex Series; droplevel + sort_index aligns
    # it back to df's integer index.
    for w in [3, 5]:
        df[f'LapTime_rolling_mean_{w}'] = (
            df.groupby(STINT_KEYS)['LapTime (s)']
              .rolling(w, min_periods=1)
              .mean()
              .droplevel(list(range(len(STINT_KEYS))))
              .sort_index()
        )
        df[f'LapTime_rolling_std_{w}'] = (
            df.groupby(STINT_KEYS)['LapTime (s)']
              .rolling(w, min_periods=2)
              .std()
              .droplevel(list(range(len(STINT_KEYS))))
              .sort_index()
              .fillna(0.0)
        )

    # Degradation slope: (current − value w−1 laps ago) / (w−1), within Driver's stint
    # groupby.shift() is C-level — fast.
    deg_grp = df.groupby(STINT_KEYS)['Cumulative_Degradation']
    for w in [3, 5]:
        slope = (df['Cumulative_Degradation'] - deg_grp.shift(w - 1)) / (w - 1)
        df[f'Degradation_rolling_slope_{w}'] = slope.fillna(0.0)

    # ── 7. Interaction features ───────────────────────────────────────────────
    df['TyreLife_x_laps_remaining']   = df['TyreLife'] * df['laps_remaining']
    df['Degradation_x_RaceProgress']  = df['Cumulative_Degradation_winsorized'] * df['RaceProgress']
    df['Position_x_RaceProgress']     = df['Position'] * df['RaceProgress']

    return df

print('build_base_features defined.')

build_base_features defined.


In [11]:
import time
t0 = time.time()
train = build_base_features(train_raw)
test  = build_base_features(test_raw)
elapsed = time.time() - t0

new_cols = train.shape[1] - train_raw.shape[1]
print(f'Done in {elapsed:.1f}s')
print(f'Train: {train_raw.shape} → {train.shape}  ({new_cols} new features)')
print(f'Test : {test_raw.shape}  → {test.shape}')

Done in 39.1s
Train: (439140, 16) → (439140, 40)  (24 new features)
Test : (188165, 15)  → (188165, 39)


## 9. Target Encodings — Fold-Aware Only

Target encoding replaces a categorical variable with the mean of the target within each category level. It can be very powerful but leaks if computed on the full dataset before splitting: the validation fold's rows contribute to the encoding values that then predict those same rows.

**Why target-encode `Race`?** Circuit pit rates span 4× (China≈39% to Mexico≈9%). One-hot encoding would add ~25 sparse columns; a single float per row is far more efficient.

**Why target-encode `Driver`?** Drivers have systematic differences in tyre management strategies and stint lengths that go beyond what the raw features capture.

**`Driver_avg_stint_length`** is not a target encoding (it doesn't use `PitNextLap`) but depends on the training set's stint structure, so it's computed in the same fold-aware call.

**Unseen values at inference time** (races or drivers in test not in train) fall back to the global mean — a safe default.

In [12]:
def apply_target_encodings(
    train_df: pd.DataFrame,
    encode_df: pd.DataFrame,
) -> pd.DataFrame:
    """
    Compute target encodings from train_df, apply to encode_df.

    Call INSIDE each CV fold:
        train_df  = fold's training rows  (must contain PitNextLap)
        encode_df = fold's validation rows OR the held-out test set

    Never call with the full training set as train_df — that leaks the target.
    """
    global_mean = train_df['PitNextLap'].mean()
    result = encode_df.copy()

    # Race: mean pit rate per circuit
    race_enc = train_df.groupby('Race')['PitNextLap'].mean()
    result['Race_target_encoded'] = result['Race'].map(race_enc).fillna(global_mean)

    # Driver: mean pit rate per driver
    driver_enc = train_df.groupby('Driver')['PitNextLap'].mean()
    result['Driver_target_encoded'] = result['Driver'].map(driver_enc).fillna(global_mean)

    # Driver avg stint length: mean of per-stint max TyreLife, per driver
    driver_stints = (
        train_df
        .groupby(['Driver', 'Race', 'Year', 'Stint'])['TyreLife']
        .max()
        .reset_index()
        .groupby('Driver')['TyreLife']
        .mean()
    )
    global_avg_stint = driver_stints.mean()
    result['Driver_avg_stint_length'] = result['Driver'].map(driver_stints).fillna(global_avg_stint)

    return result

print('apply_target_encodings defined.')

apply_target_encodings defined.


In [13]:
# Demonstrate fold-aware target encoding on an 80/20 group split
from sklearn.model_selection import GroupShuffleSplit

groups = train['Race'].astype(str) + '_' + train['Year'].astype(str)
gss    = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
tr_idx, val_idx = next(gss.split(train, groups=groups))

demo_train = train.iloc[tr_idx]
demo_val   = train.iloc[val_idx]

demo_val_enc = apply_target_encodings(demo_train, demo_val)

print('Target encoding demo (80/20 group split):')
print(f'  Train races: {demo_train["Race"].nunique()}  |  Val races: {demo_val["Race"].nunique()}')
print('\nEncoded value distribution in val fold:')
print(demo_val_enc[['Race_target_encoded', 'Driver_target_encoded', 'Driver_avg_stint_length']]
      .describe().round(3))

# Sanity: Race_target_encoded values should correlate with actual race pit rates
race_check = (demo_val_enc
              .groupby('Race')[['Race_target_encoded', 'PitNextLap']]
              .mean()
              .sort_values('Race_target_encoded'))
corr = race_check['Race_target_encoded'].corr(race_check['PitNextLap'])
print(f'\nCorrelation of Race_target_encoded with actual val pit rate: {corr:.4f}')
if abs(corr) < 0.5:
    print('  Note: low correlation is expected in this demo — the 16 val races are entirely')
    print('  different circuits from the 26 train races (GroupShuffleSplit with only 42 total')
    print('  race-year groups). In the real 5-fold CV (Notebook 05), each fold sees 80% of all')
    print('  races in training, so the encoding transfers much better to unseen val races.')

Target encoding demo (80/20 group split):
  Train races: 26  |  Val races: 16

Encoded value distribution in val fold:
       Race_target_encoded  Driver_target_encoded  Driver_avg_stint_length
count            83072.000              83072.000                83072.000
mean                 0.199                  0.203                   20.389
std                  0.098                  0.057                    3.500
min                  0.094                  0.000                    8.000
25%                  0.143                  0.184                   18.932
50%                  0.174                  0.202                   20.547
75%                  0.222                  0.226                   21.452
max                  0.446                  0.566                   46.000

Correlation of Race_target_encoded with actual val pit rate: -0.1741
  Note: low correlation is expected in this demo — the 16 val races are entirely
  different circuits from the 26 train races (GroupShuf

## 10. Feature Summary

In [14]:
BASE_FEATURES = [
    # Tyre age
    'TyreLife_normalized_by_compound', 'TyreLife_sq',
    # Compound
    'is_wet_tyre', 'compound_ordinal',
    # Race context
    'laps_remaining', 'is_testing_session',
    # Raw context — original columns with direct strategic signal
    # Stint: how many pit stops the driver has made (1st, 2nd, 3rd stint).
    #   Captures strategy phase independently of tyre wear — a 3rd-stint driver
    #   late in the race is essentially certain not to pit again.
    # Position: standalone race position independent of the Position_x_RaceProgress
    #   interaction. A P1 driver avoids pitting to protect their lead; the
    #   interaction alone cannot recover this main effect.
    'Stint', 'Position',
    # Degradation
    'Cumulative_Degradation_winsorized', 'Degradation_rate', 'Degradation_acceleration',
    # Lag
    'LapTime_lag1', 'LapTime_lag2', 'LapTime_lag3',
    'LapTime_Delta_lag1', 'LapTime_Delta_lag2', 'LapTime_Delta_lag3',
    # Rolling
    'LapTime_rolling_mean_3', 'LapTime_rolling_mean_5',
    'LapTime_rolling_std_3',  'LapTime_rolling_std_5',
    'Degradation_rolling_slope_3', 'Degradation_rolling_slope_5',
    # Interactions
    'TyreLife_x_laps_remaining', 'Degradation_x_RaceProgress', 'Position_x_RaceProgress',
]

print(f'Base features: {len(BASE_FEATURES)}')
print(f'Total columns in train: {train.shape[1]}')

# Null check — there should be none (all NaNs are filled to 0)
nulls = train[BASE_FEATURES].isnull().sum()
if nulls.any():
    print('\nWARNING — null counts:')
    print(nulls[nulls > 0])
else:
    print('\nNull check: PASSED — no nulls in any base feature')

# Correlation with target
print('\nTop 10 features by |correlation| with PitNextLap:')
corrs = train[BASE_FEATURES].corrwith(train['PitNextLap']).abs().sort_values(ascending=False)
print(corrs.head(10).round(4))

Base features: 26
Total columns in train: 40

Null check: PASSED — no nulls in any base feature

Top 10 features by |correlation| with PitNextLap:
Degradation_x_RaceProgress           0.2463
TyreLife_sq                          0.2374
TyreLife_x_laps_remaining            0.1983
Stint                                0.1982
laps_remaining                       0.1855
compound_ordinal                     0.1783
Cumulative_Degradation_winsorized    0.1730
Position_x_RaceProgress              0.1442
LapTime_lag1                         0.0408
LapTime_rolling_mean_5               0.0378
dtype: float64


In [15]:
print('Feature statistics (train):')
print(train[BASE_FEATURES].describe().round(3).to_string())

Feature statistics (train):
       TyreLife_normalized_by_compound  TyreLife_sq  is_wet_tyre  compound_ordinal  laps_remaining  is_testing_session       Stint    Position  Cumulative_Degradation_winsorized  Degradation_rate  Degradation_acceleration  LapTime_lag1  LapTime_lag2  LapTime_lag3  LapTime_Delta_lag1  LapTime_Delta_lag2  LapTime_Delta_lag3  LapTime_rolling_mean_3  LapTime_rolling_mean_5  LapTime_rolling_std_3  LapTime_rolling_std_5  Degradation_rolling_slope_3  Degradation_rolling_slope_5  TyreLife_x_laps_remaining  Degradation_x_RaceProgress  Position_x_RaceProgress
count                       439140.000   439140.000   439140.000        439140.000      439140.000          439140.000  439140.000  439140.000                         439140.000        439140.000                439140.000    439140.000    439140.000    439140.000          439140.000          439140.000          439140.000              439140.000              439140.000             439140.000             439140.00

## 11. Save to Parquet

We save all original columns plus the 26 base features. Target encodings are excluded — they will be added during training in Notebook 05.

`Stint` and `Position` are original columns already present in the parquet via `ORIG_COLS`; the `[f for f in BASE_FEATURES if f not in ORIG_COLS]` guard in cell-27 prevents duplicating them.

Parquet is preferred over CSV because:
- Preserves dtypes (e.g., `int8` flags don't round-trip as `int64` through CSV)
- ~4–6× smaller on disk than CSV for this schema
- Faster read times in downstream notebooks

In [16]:
# Columns to keep: all originals + base features (PitNextLap only in train)
ORIG_COLS = list(train_raw.columns)
KEEP_TRAIN = ORIG_COLS + [f for f in BASE_FEATURES if f not in ORIG_COLS]
KEEP_TEST  = [c for c in KEEP_TRAIN if c != 'PitNextLap']

train[KEEP_TRAIN].to_parquet(processed_dir / 'train_features.parquet', index=False)
test[KEEP_TEST].to_parquet(processed_dir  / 'test_features.parquet',  index=False)

print(f'train_features.parquet: {len(train):,} rows × {len(KEEP_TRAIN)} cols')
print(f'test_features.parquet : {len(test):,} rows × {len(KEEP_TEST)} cols')

train_features.parquet: 439,140 rows × 40 cols
test_features.parquet : 188,165 rows × 39 cols


In [17]:
# Verification: row counts and id sets must match raw data
train_check = pd.read_parquet(processed_dir / 'train_features.parquet')
test_check  = pd.read_parquet(processed_dir / 'test_features.parquet')

assert len(train_check) == len(train_raw), f'Train row count mismatch: {len(train_check)} vs {len(train_raw)}'
assert len(test_check)  == len(test_raw),  f'Test row count mismatch: {len(test_check)} vs {len(test_raw)}'
assert set(train_check['id']) == set(train_raw['id']), 'id mismatch in train'
assert set(test_check['id'])  == set(test_raw['id']),  'id mismatch in test'

# File sizes
for name in ['train_features.parquet', 'test_features.parquet']:
    size_mb = (processed_dir / name).stat().st_size / 1e6
    print(f'{name}: {size_mb:.1f} MB')

print('\nVerification: PASSED')

train_features.parquet: 41.9 MB
test_features.parquet: 16.8 MB

Verification: PASSED


## Summary

| Category | Features | Key design choice |
|----------|----------|-------------------|
| Tyre age (2) | `TyreLife_normalized_by_compound`, `TyreLife_sq` | Cliff thresholds from EDA S-curves (SOFT=13, MED=49, HARD=61), not median stint lengths |
| Compound (2) | `is_wet_tyre`, `compound_ordinal` | Wet compounds (INTERMEDIATE, WET) get ordinal=0 + dedicated flag; dry ordinal only |
| Race context (2) | `laps_remaining`, `is_testing_session` | `laps_remaining` = 1 − RaceProgress; Testing rows separated (14.7% vs 20.2% pit rate) |
| Raw context (2) | `Stint`, `Position` | Pass-through original columns. `Stint` captures strategy phase (1st/2nd/3rd pit stop). `Position` provides main effect — the `Position_x_RaceProgress` interaction alone cannot recover it. |
| Degradation (3) | `Cumulative_Degradation_winsorized`, `Degradation_rate`, `Degradation_acceleration` | Winsorize at [−205, +122]; rate normalizes by TyreLife; acceleration = 1-lag diff within `(Race, Year, Driver, Stint)` |
| Lag (6) | `LapTime_lag{1,2,3}`, `LapTime_Delta_lag{1,2,3}` | Grouped within `(Race, Year, Driver, Stint)` — NaN at boundary → 0. Driver required: `Stint` resets to 1 per driver. |
| Rolling (6) | `LapTime_rolling_mean_{3,5}`, `std_{3,5}`, `Degradation_rolling_slope_{3,5}` | `groupby().rolling()` (C-level) not `transform(lambda ...)` (Python-level, 25× slower with 2500+ groups) |
| Interaction (3) | `TyreLife_x_laps_remaining`, `Degradation_x_RaceProgress`, `Position_x_RaceProgress` | Explicit products reduce tree depth needed to capture joint effects |
| Target encoding (3) | `Race_target_encoded`, `Driver_target_encoded`, `Driver_avg_stint_length` | **Not in parquet** — computed inside CV fold via `apply_target_encodings()` |

**Total base features saved to parquet: 26**  
**Target encoding features (fold-aware, added in Notebook 05): 3**  
**Next step:** Notebook 03 — Validation Strategy (GroupKFold by Race, leakage demonstration)